In [ ]:
from webscraping import *

In [ ]:
links = [
'https://www.flashscore.com/football/argentina/liga-profesional-2024/results/', 
'https://www.flashscore.com/football/argentina/copa-de-la-liga-profesional-2024/results/',
'https://www.flashscore.com/football/bolivia/division-profesional-2024/results/',
'https://www.flashscore.com/football/brazil/serie-a-2024/results/',
'https://www.flashscore.com/football/brazil/serie-b-2024/results/',
'https://www.flashscore.com/football/brazil/serie-c-2024/results/',
'https://www.flashscore.com/football/brazil/serie-d-2024/results/',
'https://www.flashscore.com/football/chile/primera-division-2024/results/',
'https://www.flashscore.com/football/china/super-league-2024/results/',
'https://www.flashscore.com/football/colombia/primera-a-2024/results/',
'https://www.flashscore.com/football/ecuador/liga-pro-2024/results/',
'https://www.flashscore.com/football/estonia/meistriliiga-2024/results/',
'https://www.flashscore.com/football/estonia/esiliiga-2024/results/',
'https://www.flashscore.com/football/finland/veikkausliiga-2024/results/',
'https://www.flashscore.com/football/finland/ykkonen-2024/results/',
'https://www.flashscore.com/football/iceland/besta-deild-karla-2024/results/',
'https://www.flashscore.com/football/ireland/premier-division-2024/results/',
'https://www.flashscore.com/football/ireland/division-1-2024/results/',
'https://www.flashscore.com/football/japan/j1-league-2024/results/',
'https://www.flashscore.com/football/japan/j2-league-2024/results/',
'https://www.flashscore.com/football/norway/eliteserien-2024/results/',
'https://www.flashscore.com/football/norway/obos-ligaen-2024/results/',
'https://www.flashscore.com/football/paraguay/primera-division-2024/results/',
'https://www.flashscore.com/football/peru/liga-1-2024/results/',
'https://www.flashscore.com/football/south-korea/k-league-1-2024/results/',
'https://www.flashscore.com/football/south-korea/k-league-2-2024/results/',
'https://www.flashscore.com/football/sweden/allsvenskan-2024/results/',
'https://www.flashscore.com/football/sweden/superettan-2024/results/',
'https://www.flashscore.com/football/uruguay/primera-division-2024/results/',
'https://www.flashscore.com/football/usa/mls-2024/results/',
'https://www.flashscore.com/football/venezuela/liga-futve-2024/results/',
'https://www.flashscore.com/football/south-america/copa-libertadores-2024/results/',
'https://www.flashscore.com/football/south-america/copa-sudamericana-2024/results/'
]

In [3]:
for i in links:
    
    url = i
    colecao_partidas = TinyDB('base_flashscore_2024.json')
   
    options = webdriver.ChromeOptions()
    options.add_argument('--headless')

    s = Service('./chromedriver.exe')
    driver = webdriver.Chrome(service=s, options=options)
    driver.maximize_window()
    
    try:
        driver.get(url)

        if ('No match found.' in driver.page_source) == False:
            try:
                WebDriverWait(driver, 8).until(EC.visibility_of_element_located((By.CSS_SELECTOR,'button#onetrust-accept-btn-handler')))
                button_cookies = driver.find_element(By.CSS_SELECTOR,'button#onetrust-accept-btn-handler')
                button_cookies.click()
            except:
                print("cookies already closed")

            while('event__more' in driver.page_source):
                driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")    
                button_more = driver.find_element(By.CSS_SELECTOR,'a.event__more.event__more--static')
                button_more.click()
                sleep(2)

            jogos = driver.find_elements(By.CSS_SELECTOR,'div.event__match--twoLine')
            id_jogos =[]
            for i in jogos:
                id_jogos.append(i.get_attribute("id")[4:])

            season = driver.find_element(By.CSS_SELECTOR,'div.heading__info').text

            print(driver.title)

            for id_jogo in tqdm(id_jogos):
                procura_duplicado = Query()
                jogo_existente = colecao_partidas.search(procura_duplicado.Id == id_jogo)
                
                if jogo_existente == []:

                    try:
                        jogo = {}

                        Dados_Jogo(id_jogo,jogo,driver)
                        Temporada(jogo,season)
                        Gols_HT_FT(id_jogo,jogo,driver)
                        Odds_Math_Odds(id_jogo,jogo,driver)
                        Odds_Over_Under(id_jogo,jogo,driver)
                        Odds_BTTS(id_jogo,jogo,driver)
                        Odds_Dupla_Chance(id_jogo,jogo,driver)
                        Odds_Handicap_Asiatico(id_jogo,jogo,driver)
                        Odds_Correct_Score(id_jogo,jogo,driver)
                        Minutos_dos_Gols(id_jogo,jogo,driver)

                        colecao_partidas.insert(jogo.copy())

                    except Exception as e:
                        print(f"Erro ao processar o jogo {id_jogo}: {e}")

                else:
                    pass
                
        driver.close()
    except:
        pass